In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")

TICKERS = ["TSLA", "BND", "SPY"]
START = "2015-01-01"
END = "2026-06-30"

In [ ]:
import yfinance as yf

def fetch_data(tickers, start, end):
    data = {}
    for t in tickers:
        df = yf.download(t, start=start, end=end, auto_adjust=False, progress=False)
        if df.empty: # type: ignore
            raise ValueError(f"No data returned for {t}")
        df.columns = df.columns.get_level_values(0) if isinstance(df.columns, pd.MultiIndex) else df.columns # type: ignore
        data[t] = df
    return data

raw_data = fetch_data(TICKERS, START, END)

for t in TICKERS:
    print(f"{t}: {raw_data[t].shape[0]} rows, {raw_data[t].index.min().date()} -> {raw_data[t].index.max().date()}")

TSLA: 2888 rows, 2015-01-02 -> 2026-06-29
BND: 2888 rows, 2015-01-02 -> 2026-06-29
SPY: 2888 rows, 2015-01-02 -> 2026-06-29


In [3]:
import os
os.makedirs("../data/raw", exist_ok=True)
for t in TICKERS:
    raw_data[t].to_csv(f"../data/raw/{t}_raw.csv")
print("Raw data saved.")

Raw data saved.


In [4]:
long_frames = []
for t in TICKERS:
    df = raw_data[t].copy()
    df["Ticker"] = t
    long_frames.append(df)
combined_long = pd.concat(long_frames).reset_index().rename(columns={"index": "Date"})
combined_long.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume,Ticker
0,2015-01-02,14.620667,14.620667,14.883333,14.217333,14.858000,71466000,TSLA
1,2015-01-05,14.006000,14.006000,14.433333,13.810667,14.303333,80527500,TSLA
2,2015-01-06,14.085333,14.085333,14.280000,13.614000,14.004000,93928500,TSLA
3,2015-01-07,14.063333,14.063333,14.318667,13.985333,14.223333,44526000,TSLA
4,2015-01-08,14.041333,14.041333,14.253333,14.000667,14.187333,51637500,TSLA


In [5]:
for t in TICKERS:
    print(f"\n--- {t} describe() ---")
    print(raw_data[t].describe().round(2))

print("\nDtypes:")
print(combined_long.dtypes)


--- TSLA describe() ---
Price  Adj Close    Close     High      Low     Open        Volume
count    2888.00  2888.00  2888.00  2888.00  2888.00  2.888000e+03
mean      148.77   148.77   151.99   145.42   148.80  1.087922e+08
std       138.90   138.90   141.85   135.87   138.98  7.082549e+07
min         9.58     9.58    10.33     9.40     9.49  1.062000e+07
25%        18.39    18.39    18.67    18.02    18.39  6.548325e+07
50%       133.44   133.44   136.05   125.83   131.50  9.033615e+07
75%       251.93   251.93   257.49   245.83   251.68  1.261204e+08
max       489.88   489.88   498.83   485.33   489.88  9.140820e+08

--- BND describe() ---
Price  Adj Close    Close     High      Low     Open       Volume
count    2888.00  2888.00  2888.00  2888.00  2888.00      2888.00
mean       66.50    79.33    79.44    79.21    79.33   4653785.80
std         4.71     5.31     5.30     5.32     5.31   3017703.95
min        58.73    68.04    68.38    67.99    68.08         0.00
25%        62.48  